In [1]:
# Experiment 4: ML Modeling & Experiment Tracking

import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load the cleaned and feature-engineered dataset

DATA_PATH = "../data/processed/air_quality_cleaned.csv"

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2763, 22)


,city,date,aqi,pm25,pm10,no2,so2,co,o3,year,...,day_of_week,day_of_year,is_weekend,aqi_lag_1,aqi_lag_3,aqi_lag_7,target_aqi,aqi_rolling_mean_3,aqi_rolling_mean_7,aqi_rolling_std_7
0,Mumbai,2019-01-08,161,88.55,173.88,133.63,183.54,1.69,152.95,2019,...,1,8,0,146.0,178.0,194.0,154.0,172.000000,197.142857,38.429280
1,Mumbai,2019-01-09,154,84.70,166.32,127.82,175.56,1.62,146.30,2019,...,2,9,0,161.0,192.0,180.0,219.0,166.333333,192.428571,40.828328
2,Mumbai,2019-01-10,219,120.45,236.52,181.77,249.66,2.30,208.05,2019,...,3,10,0,154.0,146.0,267.0,212.0,153.666667,188.714286,43.257810
3,Mumbai,2019-01-11,212,116.60,228.96,175.96,241.68,2.23,201.40,2019,...,4,11,0,219.0,161.0,223.0,200.0,178.000000,181.857143,30.786515
4,Mumbai,2019-01-12,200,110.00,216.00,166.00,228.00,2.10,190.00,2019,...,5,12,1,212.0,154.0,178.0,206.0,195.000000,180.285714,28.534858


In [3]:
# Basic dataset inspection

print("Dataset Information")
print("=" * 60)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

print("\nColumns:")
print(df.columns.tolist())

Dataset Information
Rows: 2763
Columns: 22

Missing values:
0

Duplicate rows:
0

Date range:
2019-01-08 00:00:00 to 2026-08-05 00:00:00

Columns:
['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'is_weekend', 'aqi_lag_1', 'aqi_lag_3', 'aqi_lag_7', 'target_aqi', 'aqi_rolling_mean_3', 'aqi_rolling_mean_7', 'aqi_rolling_std_7']


In [4]:
# Ensure observations are in chronological order

df = df.sort_values("date").reset_index(drop=True)

print("Chronological order verified.")
print("First date:", df["date"].min())
print("Last date:", df["date"].max())

Chronological order verified.
First date: 2019-01-08 00:00:00
Last date: 2026-08-05 00:00:00


In [5]:
# Define the forecasting target

target_column = "target_aqi"

# Features that should not be directly used as model inputs
excluded_columns = [
    "target_aqi",
    "date",
    "city"
]

feature_columns = [
    column for column in df.columns
    if column not in excluded_columns
]

X = df[feature_columns]
y = df[target_column]

print("Number of features:", X.shape[1])
print("\nFeatures:")
print(feature_columns)

print("\nTarget:", target_column)

Number of features: 19

Features:
['aqi', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'is_weekend', 'aqi_lag_1', 'aqi_lag_3', 'aqi_lag_7', 'aqi_rolling_mean_3', 'aqi_rolling_mean_7', 'aqi_rolling_std_7']

Target: target_aqi


In [7]:
# Verify that all model features are numerical

print("Feature data types:")
display(X.dtypes)

non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()

print("\nNon-numeric features:", non_numeric)

Feature data types:


aqi                     int64
pm25                  float64
pm10                  float64
no2                   float64
so2                   float64
co                    float64
o3                    float64
year                    int64
month                   int64
day                     int64
day_of_week             int64
day_of_year             int64
is_weekend              int64
aqi_lag_1             float64
aqi_lag_3             float64
aqi_lag_7             float64
aqi_rolling_mean_3    float64
aqi_rolling_mean_7    float64
aqi_rolling_std_7     float64
dtype: object


Non-numeric features: []


In [8]:
# Chronological train-test split
# No random shuffling is used because this is a next-day forecasting problem.

split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(df["date"].iloc[0], "to", df["date"].iloc[split_index - 1])

print("\nTesting period:")
print(df["date"].iloc[split_index], "to", df["date"].iloc[-1])

Training samples: 2210
Testing samples: 553

Training period:
2019-01-08 00:00:00 to 2025-01-29 00:00:00

Testing period:
2025-01-30 00:00:00 to 2026-08-05 00:00:00


In [9]:
# Evaluation function for regression models

def evaluate_model(model_name, model, X_test, y_test):
    
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    }

In [10]:
# Define baseline regression models

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )
}

baseline_results = []
trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    trained_models[name] = model

    result = evaluate_model(
        name,
        model,
        X_test,
        y_test
    )

    baseline_results.append(result)

print("\nBaseline training completed.")

Training Linear Regression...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...

Baseline training completed.


In [11]:
# Compare baseline model performance

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df = baseline_results_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

display(
    baseline_results_df.round(4)
)

,Model,MAE,RMSE,R²
0,Gradient Boosting,26.1176,35.4854,0.5830
1,Random Forest,26.2465,35.5004,0.5826
2,Linear Regression,26.4389,36.1102,0.5682
3,XGBoost,26.9298,36.3030,0.5635


In [12]:
## Hyperparameter Tuning

Random Forest and XGBoost were selected for hyperparameter tuning because they are ensemble tree-based models and provide strong nonlinear modeling capability.

GridSearchCV is used to identify hyperparameter combinations that improve regression performance. The chronological training set is used for tuning to avoid using future test observations during model selection.

SyntaxError: invalid syntax (2974163779.py, line 3)

In [13]:
# Random Forest hyperparameter tuning

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest parameters:")
print(rf_grid.best_params_)

print("\nBest CV RMSE:")
print(-rf_grid.best_score_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best Random Forest parameters:
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

Best CV RMSE:
24.249491304510524


In [15]:
# XGBoost hyperparameter tuning

xgb = XGBRegressor(
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

print("Best XGBoost parameters:")
print(xgb_grid.best_params_)

print("\nBest CV RMSE:")
print(-xgb_grid.best_score_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best XGBoost parameters:
{'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

Best CV RMSE:
23.798400955850948


In [16]:
# Evaluate tuned models on the test set

tuned_results = []

best_rf = rf_grid.best_estimator_
best_xgb = xgb_grid.best_estimator_

tuned_results.append(
    evaluate_model(
        "Tuned Random Forest",
        best_rf,
        X_test,
        y_test
    )
)

tuned_results.append(
    evaluate_model(
        "Tuned XGBoost",
        best_xgb,
        X_test,
        y_test
    )
)

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df = tuned_results_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

display(tuned_results_df.round(4))

,Model,MAE,RMSE,R²
0,Tuned Random Forest,26.1019,35.3005,0.5873
1,Tuned XGBoost,26.7518,36.0467,0.5697


In [17]:
# Combine baseline and tuned model results

comparison_results = pd.concat(
    [
        baseline_results_df,
        tuned_results_df
    ],
    ignore_index=True
)

comparison_results = comparison_results.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

display(comparison_results.round(4))

,Model,MAE,RMSE,R²
0,Tuned Random Forest,26.1019,35.3005,0.5873
1,Gradient Boosting,26.1176,35.4854,0.5830
2,Random Forest,26.2465,35.5004,0.5826
3,Tuned XGBoost,26.7518,36.0467,0.5697
4,Linear Regression,26.4389,36.1102,0.5682
5,XGBoost,26.9298,36.3030,0.5635


In [18]:
# Calculate improvement from baseline to tuned model

rf_baseline = baseline_results_df[
    baseline_results_df["Model"] == "Random Forest"
].iloc[0]

rf_tuned = tuned_results_df[
    tuned_results_df["Model"] == "Tuned Random Forest"
].iloc[0]

xgb_baseline = baseline_results_df[
    baseline_results_df["Model"] == "XGBoost"
].iloc[0]

xgb_tuned = tuned_results_df[
    tuned_results_df["Model"] == "Tuned XGBoost"
].iloc[0]

improvement_summary = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "RMSE Baseline": [rf_baseline["RMSE"], xgb_baseline["RMSE"]],
    "RMSE Tuned": [rf_tuned["RMSE"], xgb_tuned["RMSE"]],
})

improvement_summary["RMSE Improvement"] = (
    improvement_summary["RMSE Baseline"]
    - improvement_summary["RMSE Tuned"]
)

improvement_summary["Improvement (%)"] = (
    improvement_summary["RMSE Improvement"]
    / improvement_summary["RMSE Baseline"]
) * 100

display(improvement_summary.round(4))

,Model,RMSE Baseline,RMSE Tuned,RMSE Improvement,Improvement (%)
0,Random Forest,35.5004,35.3005,0.1999,0.5630
1,XGBoost,36.3030,36.0467,0.2562,0.7058


In [20]:
import mlflow
import mlflow.sklearn

print("MLflow:", mlflow.__version__)

MLflow: 2.22.0


In [21]:
## MLflow Experiment Tracking

MLflow is used to track model experiments, hyperparameters, evaluation metrics, and trained model artifacts.

The baseline and tuned regression models are logged under a single MLflow experiment to provide reproducible experiment tracking and facilitate model comparison.

SyntaxError: invalid syntax (1813151701.py, line 3)

In [22]:
# Configure MLflow experiment

mlflow.set_experiment("Mumbai_AQI_Next_Day_Forecasting")

print("MLflow experiment configured successfully.")

2026/09/10 13:47:24 INFO mlflow.tracking.fluent: Experiment with name 'Mumbai_AQI_Next_Day_Forecasting' does not exist. Creating a new experiment.


MLflow experiment configured successfully.


In [23]:
# Log baseline models to MLflow

for model_name, model in models.items():

    with mlflow.start_run(run_name=f"Baseline_{model_name}"):

        # Evaluate model on test data
        predictions = model.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        # Log model metrics
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)

        # Log model
        mlflow.sklearn.log_model(
            model,
            artifact_path="model"
        )

        # Log model name
        mlflow.set_tag("model_type", model_name)

        print(
            f"{model_name}: "
            f"RMSE={rmse:.4f}, "
            f"MAE={mae:.4f}, "
            f"R2={r2:.4f}"
        )

2026/09/10 13:48:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Regression: RMSE=36.1102, MAE=26.4389, R2=0.5682


2026/09/10 13:48:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest: RMSE=35.5004, MAE=26.2465, R2=0.5826


2026/09/10 13:48:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Gradient Boosting: RMSE=35.4854, MAE=26.1176, R2=0.5830


2026/09/10 13:48:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost: RMSE=36.3030, MAE=26.9298, R2=0.5635


In [24]:
# Log tuned models to MLflow

tuned_models = {
    "Tuned Random Forest": best_rf,
    "Tuned XGBoost": best_xgb
}

for model_name, model in tuned_models.items():

    with mlflow.start_run(run_name=model_name.replace(" ", "_")):

        predictions = model.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        # Log metrics
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)

        # Log model hyperparameters
        if model_name == "Tuned Random Forest":
            mlflow.log_params(rf_grid.best_params_)

        elif model_name == "Tuned XGBoost":
            mlflow.log_params(xgb_grid.best_params_)

        # Log model artifact
        mlflow.sklearn.log_model(
            model,
            artifact_path="model"
        )

        mlflow.set_tag("model_type", model_name)

        print(
            f"{model_name}: "
            f"RMSE={rmse:.4f}, "
            f"MAE={mae:.4f}, "
            f"R2={r2:.4f}"
        )

2026/09/10 13:48:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Tuned Random Forest: RMSE=35.3005, MAE=26.1019, R2=0.5873


2026/09/10 13:48:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Tuned XGBoost: RMSE=36.0467, MAE=26.7518, R2=0.5697


In [ ]:
## Final Model Selection

Based on evaluation on the untouched chronological test set, the Tuned Random Forest model achieved the lowest RMSE and MAE and the highest R² among the evaluated models.

Therefore, the Tuned Random Forest model is selected as the final model for next-day AQI forecasting.

In [25]:
import joblib
from pathlib import Path

# Select the best-performing model
final_model = best_rf

# Create model directory
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save final model
MODEL_PATH = MODEL_DIR / "final_aqi_random_forest.pkl"

joblib.dump(final_model, MODEL_PATH)

print(f"Final model saved to: {MODEL_PATH}")

Final model saved to: ..\models\final_aqi_random_forest.pkl


In [26]:
# Verify that the saved model can be loaded

loaded_model = joblib.load(MODEL_PATH)

print("Saved model loaded successfully.")
print("Model type:", type(loaded_model).__name__)

Saved model loaded successfully.
Model type: RandomForestRegressor


In [27]:
# Final model prediction verification

final_predictions = loaded_model.predict(X_test)

print("Final model prediction verification")
print("=" * 60)
print("Number of predictions:", len(final_predictions))
print("First 5 predictions:", np.round(final_predictions[:5], 2))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, final_predictions)), 4))
print("Test MAE:", round(mean_absolute_error(y_test, final_predictions), 4))
print("Test R²:", round(r2_score(y_test, final_predictions), 4))

Final model prediction verification
Number of predictions: 553
First 5 predictions: [149.67 173.67 140.29 172.93 186.36]
Test RMSE: 35.3005
Test MAE: 26.1019
Test R²: 0.5873


In [ ]:
## Experiment 4 Conclusion

Machine-learning regression models were developed for next-day AQI forecasting using a chronological 80:20 train-test split to preserve the temporal nature of the dataset and avoid future-data leakage.

Four baseline models were evaluated: Linear Regression, Random Forest, Gradient Boosting, and XGBoost. Gradient Boosting achieved the best baseline RMSE of 35.4854.

Random Forest and XGBoost were subsequently tuned using GridSearchCV. The tuned Random Forest achieved a test RMSE of 35.3005, MAE of 26.1019, and R² of 0.5873. The tuned XGBoost achieved a test RMSE of 36.0467, MAE of 26.7518, and R² of 0.5697.

Based on performance on the untouched chronological test set, the Tuned Random Forest was selected as the final model. It provided the lowest RMSE and MAE and the highest R² among the evaluated models. Hyperparameter tuning improved the Random Forest RMSE from 35.5004 to 35.3005.

MLflow was used to track the baseline and tuned experiments, including model metrics, parameters, and model artifacts. The selected Tuned Random Forest model was serialized and saved for subsequent use in the AQI forecasting stage.

Overall, Experiment 4 successfully completed machine-learning model development, hyperparameter tuning, experiment tracking, model comparison, and final model selection.